In [1]:
# STEP 1: IMPORT LIBRARIES
import pandas as pd
import numpy as np
import os

folder_path = r"D:\Desktop\Data Analytics\Projects\IKEA-Customer-Analytics"
print(os.listdir(folder_path))

['01_Datasets', '01_Excel Datasets', '02_PowerBI', '03_SQL', '04_Python', '05_Screenshots', '06_Documentation']


In [3]:


# STEP 2: LOAD AND PREPARE THE DATASET

import pandas as pd

file_path = r"D:\Desktop\Data Analytics\Projects\IKEA-Customer-Analytics\01_Excel Datasets\RetailCleaned.csv"

df = pd.read_csv(file_path, header=None, low_memory=False)

df.columns = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer_ID",
    "Country",
    "SourceSheet"
]

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"], errors="coerce"
)

df["Quantity"] = pd.to_numeric(
    df["Quantity"], errors="coerce"
)

df["Price"] = pd.to_numeric(
    df["Price"], errors="coerce"
)

df["Customer_ID"] = pd.to_numeric(
    df["Customer_ID"], errors="coerce"
)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())


Dataset loaded successfully!
Dataset shape: (1033036, 9)
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer_ID', 'Country', 'SourceSheet']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,SourceSheet
0,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2010-2011
1,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,2010-2011
2,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2010-2011
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2010-2011
4,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,2010-2011


In [4]:

# STEP 3: VALIDATE THE DATE RANGE

print("Earliest date:", df["InvoiceDate"].min())
print("Latest date:", df["InvoiceDate"].max())


Earliest date: 2009-12-01 07:45:00
Latest date: 2011-12-09 12:50:00


In [5]:
# STEP 4: CALCULATE RFM METRICS

rfm_data = df[
    (df["Customer_ID"].notna()) &
    (df["Quantity"] > 0)
].copy()

snapshot_date = pd.to_datetime("2011-12-10")

rfm = rfm_data.groupby("Customer_ID").agg(
    Recency=(
        "InvoiceDate",
        lambda x: (snapshot_date - x.max()).days
    ),
    Frequency=("Invoice", "nunique"),
    Monetary=(
        "Invoice",
        lambda x: (
            rfm_data.loc[x.index, "Quantity"] *
            rfm_data.loc[x.index, "Price"]
        ).sum()
    )
).reset_index()

print("RFM calculation completed!")
print("RFM dataset shape:", rfm.shape)
display(rfm.head())

RFM calculation completed!
RFM dataset shape: (5881, 4)


,Customer_ID,Recency,Frequency,Monetary
0,12346.0,325,12,77556.46
1,12347.0,2,8,4921.53
2,12348.0,75,5,2019.40
3,12349.0,18,4,4428.69
4,12350.0,310,1,334.40


In [6]:

# STEP 5: SCORE CUSTOMERS AND ASSIGN BUSINESS SEGMENTS

rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)

def segment_customer(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'
    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal'
    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At Risk'
    elif row['R_Score'] >= 4 and row['F_Score'] <= 2:
        return 'New'
    elif row['R_Score'] == 3 and row['F_Score'] <= 2:
        return 'Needs Attention'
    else:
        return 'Lost'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)
rfm['IsChurned'] = rfm['Recency'] > 90


In [7]:
 # STEP 6: PREPARE AND EXPORT THE FINAL DATASET

df_final = rfm_data.copy()

df_final["InvoiceMonth"] = (
    df_final["InvoiceDate"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

df_final["Revenue"] = (
    df_final["Quantity"] * df_final["Price"]
)

rfm_columns = [
    "Customer_ID",
    "Recency",
    "Frequency",
    "Monetary",
    "R_Score",
    "F_Score",
    "M_Score",
    "Segment",
    "IsChurned"
]

df_final = df_final.merge(
    rfm[rfm_columns],
    on="Customer_ID",
    how="left",
    validate="many_to_one"
)

output_path = os.path.join(
    folder_path,
    "EcommerceDashboardData.xlsx"
)

df_final.to_excel(output_path, index=False)

print("Export completed successfully!")
print("Final dataset shape:", df_final.shape)
print("Saved to:", output_path)

Export completed successfully!
Final dataset shape: (779495, 19)
Saved to: D:\Desktop\Data Analytics\Projects\IKEA-Customer-Analytics\EcommerceDashboardData.xlsx


In [8]:
# STEP 7: VALIDATE THE FINAL DATASET

print("Dataset shape:", df_final.shape)
print("\nColumns:")
print(df_final.columns.tolist())

display(df_final.head())

Dataset shape: (779495, 19)

Columns:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer_ID', 'Country', 'SourceSheet', 'InvoiceMonth', 'Revenue', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'Segment', 'IsChurned']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,SourceSheet,InvoiceMonth,Revenue,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,Segment,IsChurned
0,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2010-2011,2009-12-01,30.0,157,8,2433.28,3,4,4,Loyal,True
1,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,2010-2011,2009-12-01,59.5,157,8,2433.28,3,4,4,Loyal,True
2,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2010-2011,2009-12-01,30.0,157,8,2433.28,3,4,4,Loyal,True
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2010-2011,2009-12-01,100.8,157,8,2433.28,3,4,4,Loyal,True
4,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,2010-2011,2009-12-01,39.6,157,8,2433.28,3,4,4,Loyal,True


In [9]:
# STEP 8: CUSTOMER SEGMENT SUMMARY

segment_counts = rfm["Segment"].value_counts()

print("Total unique customers:", rfm["Customer_ID"].nunique())
print("\nCustomers by segment:")
print(segment_counts)

Total unique customers: 5881

Customers by segment:
Segment
Lost               1524
Loyal              1416
Champions          1291
At Risk             821
New                 442
Needs Attention     387
Name: count, dtype: int64
